In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path().resolve().parent
RAW  = BASE / "data" / "raw"
PROC = BASE / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

print(f"RAW  : {RAW}")
print(f"PROC : {PROC}")

RAW  : C:\Users\Ridha\OneDrive\Desktop\bluestock_mf_capstone\data\raw
PROC : C:\Users\Ridha\OneDrive\Desktop\bluestock_mf_capstone\data\processed


In [2]:
nav = pd.read_csv(RAW / "02_nav_history.csv")
print(f"BEFORE — Shape: {nav.shape}, Nulls: {nav.isnull().sum().to_dict()}, Dupes: {nav.duplicated().sum()}")

nav["date"] = pd.to_datetime(nav["date"])

nav = nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

nav = nav.drop_duplicates(subset=["amfi_code", "date"])

all_funds = []
for code, group in nav.groupby("amfi_code"):
    group = group.set_index("date")
    full_idx = pd.bdate_range(group.index.min(), group.index.max())
    group = group.reindex(full_idx).ffill()
    group.index.name = "date"
    group = group.reset_index()
    group["amfi_code"] = code  # explicitly reassign amfi_code
    group = group[["amfi_code", "date", "nav"]]  # enforce column order
    all_funds.append(group)

nav = pd.concat(all_funds, ignore_index=True)

nav = nav[nav["nav"] > 0].dropna(subset=["nav"])

nav.to_csv(PROC / "clean_nav_history.csv", index=False)
print(f"AFTER  — Shape: {nav.shape}")
print(f"Columns: {list(nav.columns)}")
print(f"Sample:\n{nav.head(3)}")
print(f"Date range: {nav['date'].min()} to {nav['date'].max()}")
print("✓ Saved clean_nav_history.csv")

BEFORE — Shape: (46000, 3), Nulls: {'amfi_code': 0, 'date': 0, 'nav': 0}, Dupes: 0
AFTER  — Shape: (46000, 3)
Columns: ['amfi_code', 'date', 'nav']
Sample:
   amfi_code       date       nav
0     100016 2022-01-03  520.4608
1     100016 2022-01-04  515.0971
2     100016 2022-01-05  521.7239
Date range: 2022-01-03 00:00:00 to 2026-05-29 00:00:00
✓ Saved clean_nav_history.csv


In [3]:
txn = pd.read_csv(RAW / "08_investor_transactions.csv")
print(f"BEFORE — Shape: {txn.shape}, Nulls: {txn.isnull().sum().sum()}, Dupes: {txn.duplicated().sum()}")

txn["transaction_date"] = pd.to_datetime(txn["transaction_date"])

txn["transaction_type"] = txn["transaction_type"].str.strip().str.title()
print(f"transaction_type values: {sorted(txn['transaction_type'].unique())}")

before = len(txn)
txn = txn[txn["amount_inr"] > 0]
print(f"Removed {before - len(txn)} rows with amount <= 0")

txn["kyc_status"] = txn["kyc_status"].str.strip().str.title()
print(f"kyc_status values: {sorted(txn['kyc_status'].unique())}")

txn["city_tier"] = txn["city_tier"].str.strip().str.upper()

for col in ["state", "city", "age_group", "gender", "payment_mode"]:
    txn[col] = txn[col].str.strip()

txn.to_csv(PROC / "clean_investor_transactions.csv", index=False)
print(f"AFTER  — Shape: {txn.shape}")
print(f"Nulls: {txn.isnull().sum().to_dict()}")
print("✓ Saved clean_investor_transactions.csv")

BEFORE — Shape: (32778, 13), Nulls: 0, Dupes: 0
transaction_type values: ['Lumpsum', 'Redemption', 'Sip']
Removed 0 rows with amount <= 0
kyc_status values: ['Pending', 'Verified']
AFTER  — Shape: (32778, 13)
Nulls: {'investor_id': 0, 'transaction_date': 0, 'amfi_code': 0, 'transaction_type': 0, 'amount_inr': 0, 'state': 0, 'city': 0, 'city_tier': 0, 'age_group': 0, 'gender': 0, 'annual_income_lakh': 0, 'payment_mode': 0, 'kyc_status': 0}
✓ Saved clean_investor_transactions.csv


In [4]:
perf = pd.read_csv(RAW / "07_scheme_performance.csv")
print(f"BEFORE — Shape: {perf.shape}, Nulls: {perf.isnull().sum().sum()}, Dupes: {perf.duplicated().sum()}")

numeric_cols = ["return_1yr_pct", "return_3yr_pct", "return_5yr_pct",
                "benchmark_3yr_pct", "alpha", "beta", "sharpe_ratio",
                "sortino_ratio", "std_dev_ann_pct", "max_drawdown_pct",
                "aum_crore", "expense_ratio_pct"]
for col in numeric_cols:
    perf[col] = pd.to_numeric(perf[col], errors="coerce")

neg_sharpe = perf[perf["sharpe_ratio"] < 0]
print(f"Negative Sharpe funds: {len(neg_sharpe)}")

out_range = perf[(perf["expense_ratio_pct"] < 0.1) | (perf["expense_ratio_pct"] > 2.5)]
print(f"Expense ratio out of range: {len(out_range)}")

for col in ["scheme_name", "fund_house", "category", "plan", "risk_grade"]:
    perf[col] = perf[col].str.strip()

perf.to_csv(PROC / "clean_scheme_performance.csv", index=False)
print(f"AFTER  — Shape: {perf.shape}")
print(f"Nulls: {perf.isnull().sum().to_dict()}")
print("✓ Saved clean_scheme_performance.csv")

BEFORE — Shape: (40, 19), Nulls: 0, Dupes: 0
Negative Sharpe funds: 0
Expense ratio out of range: 0
AFTER  — Shape: (40, 19)
Nulls: {'amfi_code': 0, 'scheme_name': 0, 'fund_house': 0, 'category': 0, 'plan': 0, 'return_1yr_pct': 0, 'return_3yr_pct': 0, 'return_5yr_pct': 0, 'benchmark_3yr_pct': 0, 'alpha': 0, 'beta': 0, 'sharpe_ratio': 0, 'sortino_ratio': 0, 'std_dev_ann_pct': 0, 'max_drawdown_pct': 0, 'aum_crore': 0, 'expense_ratio_pct': 0, 'morningstar_rating': 0, 'risk_grade': 0}
✓ Saved clean_scheme_performance.csv


In [5]:
fm = pd.read_csv(RAW / "01_fund_master.csv")
print(f"BEFORE — Shape: {fm.shape}, Nulls: {fm.isnull().sum().sum()}, Dupes: {fm.duplicated().sum()}")

fm["launch_date"] = pd.to_datetime(fm["launch_date"])

str_cols = ["fund_house", "scheme_name", "category", "sub_category",
            "plan", "benchmark", "fund_manager", "risk_category", "sebi_category_code"]
for col in str_cols:
    fm[col] = fm[col].str.strip()

out_range = fm[(fm["expense_ratio_pct"] < 0.1) | (fm["expense_ratio_pct"] > 2.5)]
print(f"Expense ratio out of range: {len(out_range)}")

print(f"min_sip_amount <= 0: {(fm['min_sip_amount'] <= 0).sum()}")

fm.to_csv(PROC / "clean_fund_master.csv", index=False)
print(f"AFTER  — Shape: {fm.shape}")
print("✓ Saved clean_fund_master.csv")

BEFORE — Shape: (40, 15), Nulls: 0, Dupes: 0
Expense ratio out of range: 0
min_sip_amount <= 0: 0
AFTER  — Shape: (40, 15)
✓ Saved clean_fund_master.csv


In [6]:
aum = pd.read_csv(RAW / "03_aum_by_fund_house.csv")
print(f"BEFORE — Shape: {aum.shape}, Nulls: {aum.isnull().sum().sum()}, Dupes: {aum.duplicated().sum()}")

aum["date"] = pd.to_datetime(aum["date"])

aum = aum.sort_values(["fund_house", "date"]).reset_index(drop=True)

print(f"aum_crore <= 0: {(aum['aum_crore'] <= 0).sum()}")
print(f"aum_lakh_crore <= 0: {(aum['aum_lakh_crore'] <= 0).sum()}")

aum["fund_house"] = aum["fund_house"].str.strip()

aum.to_csv(PROC / "clean_aum_by_fund_house.csv", index=False)
print(f"AFTER  — Shape: {aum.shape}")
print("✓ Saved clean_aum_by_fund_house.csv")

BEFORE — Shape: (90, 5), Nulls: 0, Dupes: 0
aum_crore <= 0: 0
aum_lakh_crore <= 0: 0
AFTER  — Shape: (90, 5)
✓ Saved clean_aum_by_fund_house.csv


In [7]:
sip = pd.read_csv(RAW / "04_monthly_sip_inflows.csv")
print(f"BEFORE — Shape: {sip.shape}, Nulls: {sip.isnull().sum().to_dict()}, Dupes: {sip.duplicated().sum()}")

sip["month"] = pd.to_datetime(sip["month"])

sip = sip.sort_values("month").reset_index(drop=True)

null_months = sip[sip["yoy_growth_pct"].isnull()]["month"].dt.strftime("%Y-%m").tolist()
print(f"yoy_growth_pct nulls (expected — first 12 months): {null_months}")

print(f"sip_inflow_crore <= 0: {(sip['sip_inflow_crore'] <= 0).sum()}")

sip.to_csv(PROC / "clean_monthly_sip_inflows.csv", index=False)
print(f"AFTER  — Shape: {sip.shape}")
print("✓ Saved clean_monthly_sip_inflows.csv")

BEFORE — Shape: (48, 6), Nulls: {'month': 0, 'sip_inflow_crore': 0, 'active_sip_accounts_crore': 0, 'new_sip_accounts_lakh': 0, 'sip_aum_lakh_crore': 0, 'yoy_growth_pct': 12}, Dupes: 0
yoy_growth_pct nulls (expected — first 12 months): ['2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12']
sip_inflow_crore <= 0: 0
AFTER  — Shape: (48, 6)
✓ Saved clean_monthly_sip_inflows.csv


In [8]:
cat = pd.read_csv(RAW / "05_category_inflows.csv")
print(f"BEFORE — Shape: {cat.shape}, Nulls: {cat.isnull().sum().sum()}, Dupes: {cat.duplicated().sum()}")

cat["month"] = pd.to_datetime(cat["month"])

cat = cat.sort_values(["month", "category"]).reset_index(drop=True)

cat["category"] = cat["category"].str.strip()

print(f"Unique categories: {sorted(cat['category'].unique())}")
print(f"Net outflow months (negative): {(cat['net_inflow_crore'] < 0).sum()}")

cat.to_csv(PROC / "clean_category_inflows.csv", index=False)
print(f"AFTER  — Shape: {cat.shape}")
print("✓ Saved clean_category_inflows.csv")

BEFORE — Shape: (144, 3), Nulls: 0, Dupes: 0


Unique categories: ['ELSS', 'Flexi Cap', 'Gilt', 'Hybrid', 'Large & Mid Cap', 'Large Cap', 'Liquid', 'Mid Cap', 'Sectoral/Thematic', 'Short Duration', 'Small Cap', 'Value/Contra']
Net outflow months (negative): 0
AFTER  — Shape: (144, 3)
✓ Saved clean_category_inflows.csv


In [9]:
folio = pd.read_csv(RAW / "06_industry_folio_count.csv")
print(f"BEFORE — Shape: {folio.shape}, Nulls: {folio.isnull().sum().sum()}, Dupes: {folio.duplicated().sum()}")

folio["month"] = pd.to_datetime(folio["month"])

folio = folio.sort_values("month").reset_index(drop=True)

print(f"total_folios_crore <= 0: {(folio['total_folios_crore'] <= 0).sum()}")

folio["calculated_total"] = (folio["equity_folios_crore"] +
                              folio["debt_folios_crore"] +
                              folio["hybrid_folios_crore"] +
                              folio["others_folios_crore"])
folio["diff"] = (folio["total_folios_crore"] - folio["calculated_total"]).round(2)
print(f"Max difference between total and sum of parts: {folio['diff'].abs().max()}")
folio = folio.drop(columns=["calculated_total", "diff"])

folio.to_csv(PROC / "clean_industry_folio_count.csv", index=False)
print(f"AFTER  — Shape: {folio.shape}")
print("✓ Saved clean_industry_folio_count.csv")

BEFORE — Shape: (21, 6), Nulls: 0, Dupes: 0
total_folios_crore <= 0: 0
Max difference between total and sum of parts: 0.01
AFTER  — Shape: (21, 6)
✓ Saved clean_industry_folio_count.csv


In [10]:
port = pd.read_csv(RAW / "09_portfolio_holdings.csv")
print(f"BEFORE — Shape: {port.shape}, Nulls: {port.isnull().sum().sum()}, Dupes: {port.duplicated().sum()}")

port["portfolio_date"] = pd.to_datetime(port["portfolio_date"])

port["weight_pct"] = port["weight_pct"].round(2)

print(f"weight_pct <= 0: {(port['weight_pct'] <= 0).sum()}")

weight_check = port.groupby("amfi_code")["weight_pct"].sum().round(2)
print(f"Weight sum stats:\n{weight_check.describe()}")
print(f"Funds with weight > 100%: {(weight_check > 100).sum()}")

for col in ["stock_symbol", "stock_name", "sector"]:
    port[col] = port[col].str.strip()

port.to_csv(PROC / "clean_portfolio_holdings.csv", index=False)
print(f"AFTER  — Shape: {port.shape}")
print("✓ Saved clean_portfolio_holdings.csv")

BEFORE — Shape: (322, 8), Nulls: 0, Dupes: 0
weight_pct <= 0: 0
Weight sum stats:
count     34.000000
mean      99.999706
std        0.010294
min       99.980000
25%       99.990000
50%      100.000000
75%      100.007500
max      100.020000
Name: weight_pct, dtype: float64
Funds with weight > 100%: 9
AFTER  — Shape: (322, 8)
✓ Saved clean_portfolio_holdings.csv


In [11]:
bench = pd.read_csv(RAW / "10_benchmark_indices.csv")
print(f"BEFORE — Shape: {bench.shape}, Nulls: {bench.isnull().sum().sum()}, Dupes: {bench.duplicated().sum()}")

bench["date"] = pd.to_datetime(bench["date"])

bench = bench.sort_values(["index_name", "date"]).reset_index(drop=True)

print(f"close_value <= 0: {(bench['close_value'] <= 0).sum()}")

print(f"Unique indices: {sorted(bench['index_name'].unique())}")

dupe_check = bench.duplicated(subset=["date", "index_name"]).sum()
print(f"Duplicate date+index combos: {dupe_check}")

bench["index_name"] = bench["index_name"].str.strip()

bench.to_csv(PROC / "clean_benchmark_indices.csv", index=False)
print(f"AFTER  — Shape: {bench.shape}")
print("✓ Saved clean_benchmark_indices.csv")

BEFORE — Shape: (8050, 3), Nulls: 0, Dupes: 0
close_value <= 0: 0
Unique indices: ['BSE_SMALLCAP', 'CRISIL_GILT', 'CRISIL_LIQUID', 'NIFTY100', 'NIFTY50', 'NIFTY500', 'NIFTY_MIDCAP150']
Duplicate date+index combos: 0
AFTER  — Shape: (8050, 3)
✓ Saved clean_benchmark_indices.csv


In [12]:
import os

print("=" * 50)
print("DATA CLEANING SUMMARY — ALL 10 FILES")
print("=" * 50)

cleaned_files = sorted(PROC.glob("clean_*.csv"))
total_rows = 0
for f in cleaned_files:
    df = pd.read_csv(f)
    total_rows += len(df)
    print(f"✓ {f.name:<45} {df.shape[0]:>8} rows  {df.shape[1]:>3} cols  "
          f"nulls: {df.isnull().sum().sum()}")

print(f"\nTotal cleaned rows across all files: {total_rows:,}")
print(f"All files saved to: {PROC}")

DATA CLEANING SUMMARY — ALL 10 FILES
✓ clean_aum_by_fund_house.csv                         90 rows    5 cols  nulls: 0
✓ clean_benchmark_indices.csv                       8050 rows    3 cols  nulls: 0
✓ clean_category_inflows.csv                         144 rows    3 cols  nulls: 0
✓ clean_fund_master.csv                               40 rows   15 cols  nulls: 0
✓ clean_industry_folio_count.csv                      21 rows    6 cols  nulls: 0
✓ clean_investor_transactions.csv                  32778 rows   13 cols  nulls: 0
✓ clean_monthly_sip_inflows.csv                       48 rows    6 cols  nulls: 12
✓ clean_nav_history.csv                            46000 rows    3 cols  nulls: 0
✓ clean_portfolio_holdings.csv                       322 rows    8 cols  nulls: 0
✓ clean_scheme_performance.csv                        40 rows   19 cols  nulls: 0

Total cleaned rows across all files: 87,533
All files saved to: C:\Users\Ridha\OneDrive\Desktop\bluestock_mf_capstone\data\processed
